In [1]:

# EXPERIMENT 3: Amazon Braket Quantum Kernel
# Improved CICIDS2017 - Four-Feature Pilot

from pathlib import Path
import numpy as np

from braket.circuits import Circuit
from braket.devices import LocalSimulator

# Locate the project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

dataset_path = (
    project_root / "data" / "qml_pilot_local"
    / "qml_pilot_4feature_arrays.npz"
)

if not dataset_path.exists():
    raise FileNotFoundError(dataset_path)

# Load locally saved, preprocessed arrays
with np.load(dataset_path, allow_pickle=False) as data:
    X_train = data["X_train"]
    X_val = data["X_validation"]
    y_train = data["y_train"]
    y_val = data["y_validation"]
    feature_names = data["feature_names"]

assert X_train.shape == (100, 4)
assert X_val.shape == (30, 4)
assert len(y_train) == 100
assert len(y_val) == 30

print("QUANTUM DATASET LOADED")
print("----------------------")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Training labels:", np.bincount(y_train))
print("Validation labels:", np.bincount(y_val))

print("\nFeatures:")
for feature in feature_names:
    print("-", feature)

print("\nAmazon Braket environment ready.")


QUANTUM DATASET LOADED
----------------------
Training: (100, 4)
Validation: (30, 4)
Training labels: [50 50]
Validation labels: [15 15]

Features:
- Bwd Packet Length Std
- Bwd Init Win Bytes
- Packet Length Variance
- Packet Length Std

Amazon Braket environment ready.


In [2]:

# Four-qubit quantum feature map

def quantum_feature_map(features):
    features = np.asarray(features, dtype=float)

    if features.shape != (4,):
        raise ValueError("Expected exactly four features")

    circuit = Circuit()

    # Encode the four features as rotations
    for qubit, angle in enumerate(features):
        circuit.ry(qubit, float(angle))

    # Add entangling operations
    circuit.cnot(0, 1)
    circuit.cnot(1, 2)
    circuit.cnot(2, 3)

    # Second feature-dependent encoding layer
    for qubit, angle in enumerate(features):
        circuit.rz(qubit, float(angle))

    # Request exact state vector from simulator
    circuit.state_vector()

    return circuit


# Build a circuit for the first training observation
sample_circuit = quantum_feature_map(X_train[0])

print("FOUR-QUBIT QUANTUM CIRCUIT")
print("--------------------------")
print(sample_circuit)


FOUR-QUBIT QUANTUM CIRCUIT
--------------------------
T  : │     0      │  1  │     2      │     3      │     4      │
      ┌──────────┐       ┌──────────┐                           
q0 : ─┤ Ry(3.14) ├───●───┤ Rz(3.14) ├───────────────────────────
      └──────────┘   │   └──────────┘                           
      ┌──────────┐ ┌─┴─┐              ┌──────────┐              
q1 : ─┤ Ry(0.02) ├─┤ X ├──────●───────┤ Rz(0.02) ├──────────────
      └──────────┘ └───┘      │       └──────────┘              
      ┌──────────┐          ┌─┴─┐                  ┌──────────┐ 
q2 : ─┤ Ry(2.72) ├──────────┤ X ├──────────●───────┤ Rz(2.72) ├─
      └──────────┘          └───┘          │       └──────────┘ 
      ┌──────────┐                       ┌─┴─┐     ┌──────────┐ 
q3 : ─┤ Ry(2.92) ├───────────────────────┤ X ├─────┤ Rz(2.92) ├─
      └──────────┘                       └───┘     └──────────┘ 
T  : │     0      │  1  │     2      │     3      │     4      │

Additional result types: StateVecto

In [3]:

# Execute quantum feature map using Amazon Braket

simulator = LocalSimulator()

def get_quantum_state(features):
    circuit = quantum_feature_map(features)

    task = simulator.run(circuit, shots=0)
    result = task.result()

    return np.asarray(
        result.values[0], dtype=complex
    )


# Simulate two different training observations
state_1 = get_quantum_state(X_train[0])
state_2 = get_quantum_state(X_train[1])

# Compute quantum-state overlap
self_similarity = abs(
    np.vdot(state_1, state_1)
) ** 2

cross_similarity = abs(
    np.vdot(state_1, state_2)
) ** 2

print("QUANTUM STATE SIMULATION")
print("------------------------")

print("State vector size:", len(state_1))

print(
    "State normalization:",
    np.linalg.norm(state_1)
)

print(
    "Self-similarity:",
    round(float(self_similarity), 6)
)

print(
    "Cross-similarity:",
    round(float(cross_similarity), 6)
)

assert len(state_1) == 16
assert np.isclose(
    np.linalg.norm(state_1), 1.0
)
assert np.isclose(self_similarity, 1.0)
assert -1e-10 <= cross_similarity <= 1 + 1e-10

print("\nQuantum state validation: PASSED")


QUANTUM STATE SIMULATION
------------------------
State vector size: 16
State normalization: 1.0
Self-similarity: 1.0
Cross-similarity: 0.057979

Quantum state validation: PASSED


In [4]:

# EXPERIMENT 3: Quantum Kernel Matrix Construction
# Improved CICIDS2017 - Four-Feature Pilot
# Amazon Braket Local Simulator

import numpy as np
import time

print("BUILDING QUANTUM KERNEL MATRICES")
print("--------------------------------")

# Generate quantum states for training observations
start = time.perf_counter()

train_states = np.array([
    get_quantum_state(features)
    for features in X_train
])

train_state_seconds = time.perf_counter() - start

print("Training quantum states generated.")
print("Training state matrix:", train_states.shape)

# Generate quantum states for validation observations
start = time.perf_counter()

val_states = np.array([
    get_quantum_state(features)
    for features in X_val
])

val_state_seconds = time.perf_counter() - start

print("\nValidation quantum states generated.")
print("Validation state matrix:", val_states.shape)

# Verify dimensions and normalization
assert train_states.shape == (100, 16)
assert val_states.shape == (30, 16)

assert np.allclose(
    np.linalg.norm(train_states, axis=1),
    1.0,
    atol=1e-8
)

assert np.allclose(
    np.linalg.norm(val_states, axis=1),
    1.0,
    atol=1e-8
)

# Quantum fidelity kernel:
# K(x,z) = |<psi(x)|psi(z)>|^2

start = time.perf_counter()

K_train = np.abs(
    train_states @ train_states.conj().T
) ** 2

K_val = np.abs(
    val_states @ train_states.conj().T
) ** 2

kernel_seconds = time.perf_counter() - start

# Validate kernel properties
assert K_train.shape == (100, 100)
assert K_val.shape == (30, 100)

assert np.allclose(
    K_train,
    K_train.T,
    atol=1e-8
)

assert np.allclose(
    np.diag(K_train),
    1.0,
    atol=1e-8
)

# Check positive semidefiniteness
eigenvalues = np.linalg.eigvalsh(K_train)

assert eigenvalues.min() >= -1e-8

print("\nQUANTUM KERNEL RESULTS")
print("----------------------")

print("Training kernel shape:", K_train.shape)
print("Validation kernel shape:", K_val.shape)

print(
    "Training state generation time:",
    round(train_state_seconds, 4),
    "seconds"
)

print(
    "Validation state generation time:",
    round(val_state_seconds, 4),
    "seconds"
)

print(
    "Kernel computation time:",
    round(kernel_seconds, 4),
    "seconds"
)

print(
    "Minimum kernel eigenvalue:",
    round(float(eigenvalues.min()), 10)
)

print(
    "Training kernel symmetric:",
    np.allclose(K_train, K_train.T)
)

print(
    "Diagonal approximately 1:",
    np.allclose(np.diag(K_train), 1.0)
)

print("\nQuantum kernel validation: PASSED")
print("Final held-out test: NOT USED")


BUILDING QUANTUM KERNEL MATRICES
--------------------------------
Training quantum states generated.
Training state matrix: (100, 16)

Validation quantum states generated.
Validation state matrix: (30, 16)

QUANTUM KERNEL RESULTS
----------------------
Training kernel shape: (100, 100)
Validation kernel shape: (30, 100)
Training state generation time: 1.1634 seconds
Validation state generation time: 0.3415 seconds
Kernel computation time: 0.001 seconds
Minimum kernel eigenvalue: -0.0
Training kernel symmetric: True
Diagonal approximately 1: True

Quantum kernel validation: PASSED
Final held-out test: NOT USED


In [9]:

# EXPERIMENT 3: Quantum Kernel SVM Classification
# Amazon Braket - CICIDS2017 Four-Feature Pilot
# 100 training and 30 validation observations

import time
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

print("TRAINING QUANTUM KERNEL SVM")
print("---------------------------")

# Verify kernel dimensions
assert K_train.shape == (100, 100)
assert K_val.shape == (30, 100)

# Create SVM using our quantum-derived kernel
quantum_svm = SVC(
    kernel="precomputed",
    C=1.0,
    class_weight="balanced"
)

# Train the classifier
start = time.perf_counter()

quantum_svm.fit(K_train, y_train)

quantum_train_seconds = (
    time.perf_counter() - start
)

print("Quantum-kernel SVM training completed.")

# Evaluate on validation data
start = time.perf_counter()

quantum_predictions = quantum_svm.predict(
    K_val
)

quantum_inference_seconds = (
    time.perf_counter() - start
)

# Scores for ROC-AUC
quantum_scores = quantum_svm.decision_function(
    K_val
)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_val,
    quantum_predictions,
    labels=[0, 1]
).ravel()

# Calculate evaluation metrics
quantum_results = {
    "Model": "Amazon Braket Quantum Kernel SVM",
    "Training_Rows": len(y_train),
    "Validation_Rows": len(y_val),
    "Qubits": 4,
    "Accuracy": accuracy_score(
        y_val, quantum_predictions
    ),
    "Precision": precision_score(
        y_val, quantum_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val, quantum_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_val, quantum_predictions,
        zero_division=0
    ),
    "False_Positive_Rate": (
        fp / (fp + tn)
        if fp + tn > 0 else np.nan
    ),
    "ROC_AUC": roc_auc_score(
        y_val, quantum_scores
    ),
    "SVM_Training_Seconds": quantum_train_seconds,
    "SVM_Inference_Seconds": quantum_inference_seconds,
    "Kernel_Preparation_Seconds": (
        train_state_seconds
        + val_state_seconds
        + kernel_seconds
    ),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp)
}

quantum_results_df = pd.DataFrame([
    quantum_results
])

print("\nQUANTUM KERNEL SVM VALIDATION RESULTS")
print("-------------------------------------")

display(quantum_results_df.T)

print("\nQuantum-kernel classification completed.")
print("Final held-out test partition: NOT USED")


TRAINING QUANTUM KERNEL SVM
---------------------------
Quantum-kernel SVM training completed.

QUANTUM KERNEL SVM VALIDATION RESULTS
-------------------------------------


,0
Model,Amazon Braket Quantum Kernel SVM
Training_Rows,100
Validation_Rows,30
Qubits,4
Accuracy,0.9
Precision,1.0
Recall,0.8
F1,0.888889
False_Positive_Rate,0.0
ROC_AUC,0.951111



Quantum-kernel classification completed.
Final held-out test partition: NOT USED


In [6]:
%pip install pandas scikit-learn

  Using cached pandas-3.0.6-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.6-cp313-cp313-win_amd64.whl (9.6 MB)
Using cached tzdata-2026.5-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
 

In [7]:
import sys
import pandas as pd
import sklearn

print("Python environment:", sys.executable)
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("\nDependencies verified successfully.")

Python environment: c:\Users\MP25NN1Q\Downloads\Quantum Modeling\Exploring-Quantum-Computing---CICIDS2017-Dataset-Test\.venv-braket\Scripts\python.exe
Pandas version: 3.0.6
Scikit-learn version: 1.9.1

Dependencies verified successfully.


In [8]:
print("Training kernel:", K_train.shape)
print("Validation kernel:", K_val.shape)
print("Training labels:", len(y_train))
print("Validation labels:", len(y_val))

Training kernel: (100, 100)
Validation kernel: (30, 100)
Training labels: 100
Validation labels: 30


In [10]:

# EXPERIMENT 3: Preserve Quantum ML Results
# Improved CICIDS2017 - Amazon Braket
# Classical vs Quantum Kernel SVM

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import numpy as np
import pandas as pd
import sklearn

# Locate project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Load previously saved classical benchmark
classical_file = (
    results_dir /
    "qml_pilot_classical_rbf_validation.csv"
)

classical_df = pd.read_csv(classical_file)

assert len(classical_df) == 1
assert len(quantum_results_df) == 1

# Save quantum classifier results
quantum_file = (
    results_dir /
    "qml_pilot_quantum_kernel_svm_validation.csv"
)

quantum_results_df.to_csv(
    quantum_file,
    index=False
)

# Prepare direct metric comparison
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "False_Positive_Rate",
    "ROC_AUC",
    "TN",
    "FP",
    "FN",
    "TP"
]

comparison = pd.DataFrame({
    "Metric": metrics,
    "Classical_RBF_SVM": [
        classical_df.iloc[0][m]
        for m in metrics
    ],
    "Quantum_Kernel_SVM": [
        quantum_results_df.iloc[0][m]
        for m in metrics
    ]
})

comparison["Quantum_Minus_Classical"] = (
    comparison["Quantum_Kernel_SVM"]
    - comparison["Classical_RBF_SVM"]
)

comparison_file = (
    results_dir /
    "qml_pilot_classical_vs_quantum_validation.csv"
)

comparison.to_csv(
    comparison_file,
    index=False
)

# Calculate recorded runtimes
classical_total = (
    float(classical_df.iloc[0]["Training_Seconds"])
    + float(classical_df.iloc[0]["Inference_Seconds"])
)

quantum_total = (
    float(quantum_results["Kernel_Preparation_Seconds"])
    + float(quantum_results["SVM_Training_Seconds"])
    + float(quantum_results["SVM_Inference_Seconds"])
)

# Link results to the exact local dataset
dataset_path = (
    project_root / "data" / "qml_pilot_local"
    / "qml_pilot_4feature_arrays.npz"
)

dataset_hash = hashlib.sha256(
    dataset_path.read_bytes()
).hexdigest()

metadata = {
    "experiment": "CICIDS2017 Quantum Kernel SVM Pilot",
    "recorded_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": "Improved CICIDS2017",
    "dataset_sha256": dataset_hash,
    "training_rows": int(len(y_train)),
    "validation_rows": int(len(y_val)),
    "selected_features": feature_names.tolist(),
    "qubits": 4,
    "simulator": "Amazon Braket LocalSimulator",
    "simulation_type": "Exact state vector",
    "shots": 0,
    "feature_map": (
        "RY encoding, CNOT chain, RZ encoding"
    ),
    "quantum_kernel": (
        "Squared quantum state overlap"
    ),
    "classical_model": "RBF SVM",
    "quantum_model": "Precomputed-kernel SVM",
    "svm_C": 1.0,
    "class_weight": "balanced",
    "classical_gamma": "scale",
    "validation_class_distribution": {
        "benign": 15,
        "malicious": 15
    },
    "classical_recorded_seconds": classical_total,
    "quantum_recorded_seconds": quantum_total,
    "quantum_kernel_min_eigenvalue": float(
        np.linalg.eigvalsh(K_train).min()
    ),
    "braket_sdk_version": version(
        "amazon-braket-sdk"
    ),
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "final_test_used": False,
    "limitations": [
        "Small balanced validation sample",
        "No quantum hardware execution",
        "No demonstrated quantum advantage",
        "No independent unseen-attack evaluation",
        "Runtime implementations differ"
    ]
}

metadata_file = (
    results_dir /
    "qml_pilot_quantum_kernel_metadata.json"
)

with open(
    metadata_file, "w", encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4)

print("QUANTUM EXPERIMENT EVIDENCE SAVED")
print("---------------------------------")

print("Quantum results:", quantum_file)
print("Model comparison:", comparison_file)
print("Metadata:", metadata_file)

print("\nCLASSICAL VS QUANTUM COMPARISON")
display(comparison.round(6))

print("\nRecorded execution totals:")
print("Classical:", round(classical_total, 6), "sec")
print("Quantum:", round(quantum_total, 6), "sec")

print("\nFinal held-out test: NOT USED")
print("Evidence preservation completed.")


QUANTUM EXPERIMENT EVIDENCE SAVED
---------------------------------
Quantum results: c:\Users\MP25NN1Q\Downloads\Quantum Modeling\Exploring-Quantum-Computing---CICIDS2017-Dataset-Test\results\qml_pilot_quantum_kernel_svm_validation.csv
Model comparison: c:\Users\MP25NN1Q\Downloads\Quantum Modeling\Exploring-Quantum-Computing---CICIDS2017-Dataset-Test\results\qml_pilot_classical_vs_quantum_validation.csv
Metadata: c:\Users\MP25NN1Q\Downloads\Quantum Modeling\Exploring-Quantum-Computing---CICIDS2017-Dataset-Test\results\qml_pilot_quantum_kernel_metadata.json

CLASSICAL VS QUANTUM COMPARISON


,Metric,Classical_RBF_SVM,Quantum_Kernel_SVM,Quantum_Minus_Classical
0,Accuracy,0.900000,0.900000,0.000000
1,Precision,1.000000,1.000000,0.000000
2,Recall,0.800000,0.800000,0.000000
3,F1,0.888889,0.888889,0.000000
4,False_Positive_Rate,0.000000,0.000000,0.000000
5,ROC_AUC,0.946667,0.951111,0.004444
6,TN,15.000000,15.000000,0.000000
7,FP,0.000000,0.000000,0.000000
8,FN,3.000000,3.000000,0.000000
9,TP,12.000000,12.000000,0.000000



Recorded execution totals:
Classical: 0.005654 sec
Quantum: 1.522033 sec

Final held-out test: NOT USED
Evidence preservation completed.
